script to examine statistics of the 2024-2025 database of hazards information

produce:
- temporal coverage 
- number of observations map (CONUS)
- histograms of the variables


known data coverage (see decisions.md)
complete wildfire and terrestrial-weather daily coverage for 2024-2025 (731/731 days each). Space weather has 677/731 days, with gaps from 2024-06-17 through 2024-08-05 and 2025-02-06 through 2025-02-09. A preliminary full-period run may proceed with those space-weather gaps explicitly flagged, but missing data must not be converted into zero hazard.

In [ ]:
from pathlib import Path
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

ROOT = Path("/Users/ryanmc/Documents/NASA_JPL/Projects/NaturalHazards/NASA ROSES Disasters 2025-2027/data")
WILDFIRE_DIR = ROOT / "wildfire" / "daily"
TERRESTRIAL_DIR = ROOT / "terrestrial_weather"
SPACE_WEATHER_DIR = ROOT / "space_weather" / "NOAA_geoE"
GRID_PATH = ROOT / "analysis_grid_CONUS_50km.nc"

START = pd.Timestamp("2024-01-01")
END = pd.Timestamp("2025-12-31")
EXPECTED_DATES = pd.date_range(START, END, freq="D")
OUTPUT_DIR = Path("outputs") / "hazards_statistics_2024_2025"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 20)

print(f"Analysis period: {START.date()} to {END.date()} ({len(EXPECTED_DATES):,} days)")
print(f"Grid file exists: {GRID_PATH.exists()}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

## 1. File coverage and daily observation counts

The tables below compare the files found with the expected 731-day period. Counts are based on finite values, with known negative MRMS sentinels and the geoelectric NetCDF fill value excluded. A missing day remains missing; it is never treated as a zero hazard day.

In [ ]:
def date_from_name(path):
    match = re.search(r"(20\d{6})", path.name)
    return pd.to_datetime(match.group(1), format="%Y%m%d") if match else pd.NaT


def dated_files(directory, pattern):
    records = [(date_from_name(path), path) for path in directory.glob(pattern)]
    return sorted((date, path) for date, path in records if pd.notna(date) and START <= date <= END)


def coverage_table(file_records, label):
    found = pd.Series({date: path.name for date, path in file_records}, dtype="string")
    table = pd.DataFrame({"date": EXPECTED_DATES})
    table["file"] = table["date"].map(found)
    table["available"] = table["file"].notna()
    table["source"] = label
    return table

wildfire_files = dated_files(WILDFIRE_DIR, "fires_analysis_grid_*.nc")
terrestrial_files = dated_files(TERRESTRIAL_DIR, "weather_CONUS_*.nc")
space_files = dated_files(SPACE_WEATHER_DIR, "*.nc")

coverage = pd.concat(
    [coverage_table(wildfire_files, "wildfire"),
     coverage_table(terrestrial_files, "terrestrial_weather"),
     coverage_table(space_files, "space_weather")],
    ignore_index=True,
)
coverage_summary = coverage.groupby("source")["available"].agg(["sum", "count"])
coverage_summary["missing"] = coverage_summary["count"] - coverage_summary["sum"]
coverage_summary["coverage_pct"] = 100 * coverage_summary["sum"] / coverage_summary["count"]
print(coverage_summary.round(2))

for source, group in coverage.groupby("source"):
    missing = group.loc[~group["available"], "date"]
    print(f"\n{source}: {len(missing)} missing days")
    if len(missing):
        runs = missing.diff().ne(pd.Timedelta(days=1)).cumsum()
        print(missing.groupby(runs).agg(["min", "max"]).to_string(index=False))

coverage.to_csv(OUTPUT_DIR / "daily_file_coverage.csv", index=False)
coverage_summary.to_csv(OUTPUT_DIR / "coverage_summary.csv")

## 2. Spatial observation coverage

For gridded products, `observed_days` is the number of daily files containing at least one valid value in each 50 km cell. For geoelectric data, the map counts valid reporting points assigned to their nearest analysis-grid cell, using one time slice per available day.

In [ ]:
from scipy.spatial import cKDTree

def grid_lat_lon(grid):
    with xr.open_dataset(grid) as ds:
        lat_name = "lat" if "lat" in ds else "latitude"
        lon_name = "lon" if "lon" in ds else "longitude"
        return ds[lat_name].values, ds[lon_name].values


def clean_values(values, variable=None):
    values = np.asarray(values, dtype="float64")
    invalid = ~np.isfinite(values)
    if variable and any(token in variable.lower() for token in ("mesh", "echotop", "nldn", "reflect", "vil", "vii")):
        invalid |= values < 0
    if variable in {"Ex", "Ey", "emag"}:
        invalid |= np.abs(values) > 1e6
    return np.where(invalid, np.nan, values)


def update_grid_counts(records, variable):
    counts = None
    for _, path in records:
        with xr.open_dataset(path) as ds:
            if variable not in ds:
                continue
            values = clean_values(ds[variable].values, variable)
            daily_count = np.isfinite(values).any(axis=0) if values.ndim == 3 else np.isfinite(values)
            counts = daily_count.astype("int32") if counts is None else counts + daily_count
    return counts

analysis_lat, analysis_lon = grid_lat_lon(GRID_PATH)
terrestrial_counts = {
    variable: update_grid_counts(terrestrial_files, variable)
    for variable in ["t2m", "i10fg", "MultiSensor_QPE_01H_Pass2_00.00"]
}
wildfire_counts = update_grid_counts(wildfire_files, "frp")

# Assign one valid geoelectric time slice from each daily file to the nearest grid cell.
flat_grid = np.column_stack([analysis_lat.ravel(), analysis_lon.ravel()])
point_tree = cKDTree(flat_grid)
space_point_counts = np.zeros(flat_grid.shape[0], dtype="int32")
space_valid_points_by_day = []
for date, path in space_files:
    with xr.open_dataset(path) as ds:
        lat = np.asarray(ds["latitude"].values).ravel()
        lon = np.asarray(ds["longitude"].values).ravel()
        ex = clean_values(ds["Ex"].isel(time=0).values, "Ex").ravel()
        valid = np.isfinite(lat) & np.isfinite(lon) & np.isfinite(ex)
        _, nearest = point_tree.query(np.column_stack([lat[valid], lon[valid]]))
        np.add.at(space_point_counts, nearest, 1)
        space_valid_points_by_day.append((date, valid.sum()))
space_point_counts = space_point_counts.reshape(analysis_lat.shape)

spatial_summary = pd.DataFrame([
    {"product": "wildfire_frp", "cells_with_observations": np.count_nonzero(wildfire_counts), "total_cell_days": np.nansum(wildfire_counts), "max_days_per_cell": np.nanmax(wildfire_counts)},
    *[{"product": variable, "cells_with_observations": np.count_nonzero(counts), "total_cell_days": np.nansum(counts), "max_days_per_cell": np.nanmax(counts)} for variable, counts in terrestrial_counts.items()],
    {"product": "space_weather_valid_points", "cells_with_observations": np.count_nonzero(space_point_counts), "total_cell_days": np.nansum(space_point_counts), "max_days_per_cell": np.nanmax(space_point_counts)},
])
print(spatial_summary)
spatial_summary.to_csv(OUTPUT_DIR / "spatial_observation_summary.csv", index=False)

fig, axes = plt.subplots(1, 5, figsize=(20, 4), constrained_layout=True)
map_data = [("Wildfire FRP", wildfire_counts), *[(name, values) for name, values in terrestrial_counts.items()], ("GeoE points", space_point_counts)]
for axis, (title, values) in zip(axes, map_data):
    image = axis.pcolormesh(analysis_lon, analysis_lat, values, shading="auto", cmap="viridis")
    axis.set_title(title, fontsize=10)
    axis.set_xlabel("Longitude")
    axis.set_ylabel("Latitude")
    fig.colorbar(image, ax=axis, shrink=0.8)
fig.suptitle("Valid observations assigned to the CONUS 50 km grid")
fig.savefig(OUTPUT_DIR / "observation_coverage_maps.png", dpi=160)
plt.show()

## 3. Variable distributions and data quality

These summaries use a regular two-week sample of available days to keep notebook execution interactive. Percentiles are calculated after masking non-finite values, known negative radar sentinels, and geoelectric fill-value artifacts. The raw negative share is retained as a quality indicator.

In [ ]:
CORE_VARIABLES = {
    "t2m": "temperature (K)",
    "d2m": "dew point (K)",
    "wspd10": "10 m wind speed (m/s)",
    "i10fg": "10 m wind gust (m/s)",
    "MultiSensor_QPE_01H_Pass2_00.00": "hourly precipitation (mm)",
    "MESH_00.50": "hail estimate (in)",
    "EchoTop_18_00.50": "echo top (km)",
    "NLDN_CG_001min_AvgDensity_00.00": "lightning density",
}

sampled_terrestrial = terrestrial_files[::14]
summary_rows = []
histogram_values = {}
for variable, label in CORE_VARIABLES.items():
    raw_chunks = []
    for _, path in sampled_terrestrial:
        with xr.open_dataset(path) as ds:
            if variable in ds:
                raw_chunks.append(np.asarray(ds[variable].values).ravel())
    if not raw_chunks:
        continue
    raw = np.concatenate(raw_chunks)
    cleaned = clean_values(raw, variable)
    finite = cleaned[np.isfinite(cleaned)]
    summary_rows.append({
        "variable": variable,
        "description": label,
        "sampled_days": len(raw_chunks),
        "raw_values": len(raw),
        "finite_pct": 100 * np.isfinite(cleaned).mean(),
        "raw_negative_pct": 100 * (raw < 0).mean(),
        "min": np.min(finite) if len(finite) else np.nan,
        "p01": np.percentile(finite, 1) if len(finite) else np.nan,
        "median": np.percentile(finite, 50) if len(finite) else np.nan,
        "p99": np.percentile(finite, 99) if len(finite) else np.nan,
        "max": np.max(finite) if len(finite) else np.nan,
    })
    histogram_values[variable] = finite

# GeoE magnitude is derived only after fill-value masking.
geo_chunks = []
for _, path in space_files[::14]:
    with xr.open_dataset(path) as ds:
        ex = clean_values(ds["Ex"].values, "Ex")
        ey = clean_values(ds["Ey"].values, "Ey")
        geo_chunks.append(np.hypot(ex, ey).ravel())
geo_values = np.concatenate(geo_chunks)
geo_values = geo_values[np.isfinite(geo_values)]
summary_rows.append({
    "variable": "|E|",
    "description": "geoelectric field magnitude (mV/km)",
    "sampled_days": len(space_files[::14]),
    "raw_values": sum(chunk.size for chunk in geo_chunks),
    "finite_pct": 100 * len(geo_values) / sum(chunk.size for chunk in geo_chunks),
    "raw_negative_pct": np.nan,
    "min": np.min(geo_values),
    "p01": np.percentile(geo_values, 1),
    "median": np.percentile(geo_values, 50),
    "p99": np.percentile(geo_values, 99),
    "max": np.max(geo_values),
})
histogram_values["|E|"] = geo_values

variable_summary = pd.DataFrame(summary_rows).set_index("variable")
print(variable_summary.round(3).to_string())
variable_summary.to_csv(OUTPUT_DIR / "variable_summary_statistics.csv")

plot_variables = list(histogram_values)
fig, axes = plt.subplots(3, 3, figsize=(14, 11), constrained_layout=True)
for axis, variable in zip(axes.flat, plot_variables):
    values = histogram_values[variable]
    upper = np.percentile(values, 99.5) if len(values) else 1
    axis.hist(values[values <= upper], bins=50, color="#176b87", alpha=0.85)
    axis.set_title(variable, fontsize=10)
    axis.set_xlabel("value (clipped at sample p99.5)")
    axis.set_ylabel("count")
for axis in axes.flat[len(plot_variables):]:
    axis.set_visible(False)
fig.suptitle("2024-2025 hazard-variable distributions (two-week sample)")
fig.savefig(OUTPUT_DIR / "hazard_variable_histograms.png", dpi=160)
plt.show()

## 4. Temporal coverage overview

The availability panel shows whether a daily source file exists. The geoelectric series adds the number of valid reporting points in the first hourly slice of each available day, making the space-weather gap and any low-observation days visible without interpreting missing data as quiet conditions.

In [ ]:
space_daily = pd.DataFrame(space_valid_points_by_day, columns=["date", "valid_points"]).set_index("date")
coverage_plot = coverage.pivot(index="date", columns="source", values="available").astype(float)

fig, axes = plt.subplots(2, 1, figsize=(15, 6), sharex=True, constrained_layout=True)
coverage_plot.plot.area(ax=axes[0], stacked=False, linewidth=0, alpha=0.7, color=["#176b87", "#e07a5f", "#3a936f"])
axes[0].set_ylabel("daily file present")
axes[0].set_yticks([0, 1])
axes[0].set_yticklabels(["missing", "available"])
axes[0].set_title("Daily source-file coverage")
axes[0].legend(loc="lower left", ncol=3)
space_daily["valid_points"].plot(ax=axes[1], color="#c44536", linewidth=0.8)
axes[1].set_ylabel("valid points")
axes[1].set_title("GeoE valid reporting points in first hourly slice")
axes[1].set_xlabel("date")
fig.savefig(OUTPUT_DIR / "temporal_coverage.png", dpi=160)
plt.show()

# Keep units explicit in the exported spatial summary.
spatial_summary_units = spatial_summary.copy()
spatial_summary_units["metric"] = ["cell-days" if product != "space_weather_valid_points" else "point observations" for product in spatial_summary_units["product"]]
spatial_summary_units = spatial_summary_units.rename(columns={"max_days_per_cell": "max_count_per_cell"})
spatial_summary_units.to_csv(OUTPUT_DIR / "spatial_observation_summary.csv", index=False)
print("\nSaved analysis tables and figures to:", OUTPUT_DIR.resolve())